# 1. Cấu hình

Notebook này clone mã nguồn công khai từ nhánh main; không cần token. Chỉ chạy baseline M3 trên GPU Colab, với dev cases và system prompt đã có trong repo. Trước khi chạy, đẩy commit muốn dùng lên nhánh main; có thể cố định commit bằng biến bên dưới. Outputs lưu vào Google Drive theo mặc định.


In [1]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/coder-fishing/KidStory.git"
BRANCH = "main"
COMMIT_TO_RUN = None  # Có thể đặt SHA đã push để khóa chính xác mã nguồn.
REPO_DIR = Path("/content/kids-story")
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/kids-story-runs/m3")
RUNTIME_OUTPUT_ROOT = Path("/content/outputs/m3")


# 2. Tải mã nguồn công khai

Repo hiện public, nên Colab clone trực tiếp từ nhánh `main`; không cần GitHub Secret hay token. Cell ghi commit và xác nhận working tree sạch.


In [2]:
import subprocess

if REPO_DIR.exists():
    raise RuntimeError(f"Thư mục {REPO_DIR} đã tồn tại; hãy dùng runtime mới để tránh dùng nhầm mã nguồn.")
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPOSITORY_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if clone.returncode:
    raise RuntimeError("Clone thất bại. Kiểm tra kết nối mạng và nhánh main.")
source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True,
).stdout.strip()
source_status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_DIR, check=True, capture_output=True, text=True,
).stdout
if source_status.strip():
    raise RuntimeError("Repo clone không sạch; dừng để tránh chạy mã nguồn đã bị sửa.")
print(f"Đã lấy mã nguồn commit {source_commit}; working tree sạch.")


Đã lấy mã nguồn commit b0e30170fddff79bd8760b8abe496e73b6b8402e; working tree sạch.


# 3. Kiểm tra GPU Colab

Cell này kiểm tra runtime trước khi đọc dev cases hoặc tải model. Nếu không có GPU CUDA, dừng tại đây; không tự chuyển sang CPU.


In [3]:
import platform
try:
    import torch
except ImportError as exc:
    raise RuntimeError('Colab PyTorch is missing; select a standard Colab Python runtime.') from exc

torch_before_install = torch.__version__
runtime_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
}
if not runtime_info["cuda_available"]:
    raise RuntimeError("Không phát hiện GPU CUDA. Trong Colab chọn Runtime > Change runtime type > GPU rồi chạy lại.")
runtime_info["gpu_name"] = torch.cuda.get_device_name(0)
runtime_info["gpu_memory_bytes"] = int(torch.cuda.get_device_properties(0).total_memory)
print(runtime_info)


{'python': '3.13.16', 'torch': '2.14.1+cu130', 'cuda_build': '13.0', 'cuda_available': True, 'gpu_name': 'Tesla T4', 'gpu_memory_bytes': 15637086208}


# 4. Cài dependencies

PyTorch do Colab cung cấp được giữ nguyên. Chỉ cài các dependency inference đã pin; Unsloth thuộc M4, không dùng trong notebook M3 này.


In [4]:
%pip install -r /content/kids-story/requirements/m3-colab.lock.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.1 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [accelerate]

In [5]:
import importlib.metadata

expected_versions = {"transformers": "4.57.3", "accelerate": "1.12.0"}
actual_versions = {}
for name in expected_versions:
    try:
        actual_versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f'Required package missing: {name}. Re-run the dependency installation cell.') from exc
if actual_versions != expected_versions:
    raise RuntimeError(f"Phiên bản dependency không khớp lock: {actual_versions}")
if torch.__version__ != torch_before_install:
    raise RuntimeError(f"PyTorch changed during dependency install: {torch_before_install} -> {torch.__version__}")
print({"installed": actual_versions, "torch_kept_from_colab": torch.__version__})


{'installed': {'transformers': '4.57.3', 'accelerate': '1.12.0'}, 'torch_kept_from_colab': '2.14.1+cu130'}


# 5. Chạy baseline

Mặc định notebook mount Google Drive bằng giao diện Colab để giữ artifact sau khi runtime kết thúc. Nếu tắt tùy chọn đó, output chỉ nằm trong runtime tạm thời. Mỗi lần chạy tạo thư mục mới theo UTC; runner từ chối ghi đè.


In [6]:
import hashlib
import json
import sys
from datetime import datetime, timezone

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_parent = DRIVE_OUTPUT_ROOT
else:
    output_parent = RUNTIME_OUTPUT_ROOT
output_parent.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("qwen25_3b_%Y%m%dT%H%M%SZ")
output_dir = output_parent / run_id

sys.path.insert(0, str(REPO_DIR / "src"))
from kids_story.baseline import MODEL_ID, MODEL_REVISION, run_baseline
if MODEL_ID != 'Qwen/Qwen2.5-3B-Instruct' or MODEL_REVISION != 'aa8e72537993ba99e69dfaafa59ed015b17504d1':
    raise RuntimeError('Pinned model ID/revision do not match the approved M3 specification.')

input_path = REPO_DIR / "data/evaluation/dev_cases.jsonl"
system_prompt_path = REPO_DIR / "prompts/story_system_v1.txt"
result = run_baseline(
    input_path,
    system_prompt_path,
    output_dir,
    source_commit=source_commit,
)

complete_path = output_dir / "COMPLETE.json"
completion = json.loads(complete_path.read_text(encoding="utf-8"))
for filename, expected_sha256 in completion["files"].items():
    actual_sha256 = hashlib.sha256((output_dir / filename).read_bytes()).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Hash artifact không khớp: {filename}")
print({"run_id": run_id, "status": result["status"], "case_count": result["total_cases"], "artifact_dir": str(output_dir), "hashes_verified": True})
if not SAVE_TO_DRIVE:
    print("CẢNH BÁO: artifacts chỉ nằm trong runtime Colab tạm thời; hãy tải/copy chúng trước khi kết thúc phiên.")


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'run_id': 'qwen25_3b_20261116T045620Z', 'status': 'complete', 'case_count': 20, 'artifact_dir': '/content/drive/MyDrive/kids-story-runs/m3/qwen25_3b_20261116T045620Z', 'hashes_verified': True}


## Giới hạn lần chạy

Chạy thành công chỉ tạo artifact baseline trên dev; không chứng minh chất lượng, an toàn hoặc cải thiện so với fine-tune. Không dùng golden draft để chỉnh prompt/chọn model. M2 vẫn có các cổng duyệt riêng đang chờ.
